# 01 — Supervisor: clasificación de intención

**Objetivo:** entender cómo el agente Supervisor convierte un pedido en lenguaje
natural en los parámetros estructurados que consume el resto del pipeline.

**Qué vas a ver:**
1. El **núcleo puro** `clasificar_intencion`, que recibe un generador inyectado.
2. El **nodo del grafo** `construir_nodo_supervisor`, que lo envuelve y escribe
   las 5 claves del `AgentState`.
3. El **fallback de tema** y el **rechazo de salidas inválidas**.
4. (Opcional) la misma clasificación con un LLM real, si tienes proveedor.

**Qué vas a aprender:** por qué el agente no instancia el LLM por su cuenta, y
cómo el mismo núcleo corre con un doble (`FakeGenerator`) o con el modelo real
sin cambiar una línea de lógica.


In [ ]:
import sys
from pathlib import Path

# El notebook corre desde notebooks/; el código vive en la raíz del repo.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
# `src/` tambien entra al path porque `llm_provider` importa `config` de forma
# "pelada" (gotcha del AGENTS.md). Es un puente para el notebook, no el estilo.
sys.path.insert(0, str(RAIZ / "src"))

print("Raiz del repo:", RAIZ)


## 1. El núcleo puro y su dependencia inyectada

`clasificar_intencion` no sabe si detrás hay Gemini, DeepSeek o un doble de
test. Solo espera un objeto con `async generate(prompt, output_model)` — ese es
el protocolo `GeneradorEstructurado`.


In [ ]:
from typing import Any

from pydantic import BaseModel

from src.agentes.agente_supervisor import IntencionOut, clasificar_intencion


class FakeGenerator:
    """Doble que cumple `GeneradorEstructurado` y no toca ningún proveedor."""

    def __init__(self, salida: Any):
        self.salida = salida
        self.prompt = ""
        self.output_model: type[BaseModel] | None = None

    async def generate(self, *, prompt: str, output_model: type[BaseModel]) -> Any:
        self.prompt = prompt
        self.output_model = output_model
        if isinstance(self.salida, Exception):
            raise self.salida
        return self.salida


In [ ]:
generador = FakeGenerator(
    {
        "tema_consulta": "VCN",
        "perfil_destinatario": "Principiante",
        "formato_salida": "Flashcards",
        "nicho_sector": "General",
        "nivel_detalle": "Didactico",
    }
)

intencion = await clasificar_intencion("Quiero aprender VCN", None, generador)
print(intencion.model_dump())


## 2. El nodo del grafo

`construir_nodo_supervisor(generador)` devuelve una función que LangGraph puede
colgar del grafo. Lee el último mensaje del usuario desde el estado y escribe
**5 claves** (`tema_consulta`, `perfil_destinatario`, `formato_salida`,
`nicho_sector`, `nivel_detalle`), con los enums en su **forma canónica corta**
(`"Principiante"`, no la forma larga).


In [ ]:
from src.agentes.supervisor import construir_nodo_supervisor

nodo = construir_nodo_supervisor(generador)

salida = await nodo({"mensajes": [{"role": "user", "content": "Quiero aprender VCN"}]})
print(salida)


## 3. Fallback de tema

Si el modelo no extrae un `tema_consulta`, el agente usa `tema_pedido_chat`
(lo que el usuario ya escribió para el buscador). Son campos independientes:
es solo una conveniencia, no una unificación.


In [ ]:
generador_sin_tema = FakeGenerator({"tema_consulta": None})
nodo_sin_tema = construir_nodo_supervisor(generador_sin_tema)

salida = await nodo_sin_tema(
    {
        "mensajes": [{"role": "user", "content": "hola"}],
        "tema_pedido_chat": "redes VCN",
    }
)
print("tema_consulta =", salida["tema_consulta"])


## 4. Las salidas inválidas se rechazan

`IntencionOut` tiene `extra="forbid"`: un campo o valor fuera del contrato es un
error, no se ignora. Eso es lo que permite confiar en la salida del modelo.


In [ ]:
from src.errores import ErrorSalidaInvalida

generador_malo = FakeGenerator({"perfil_destinatario": "Astronauta"})
nodo_malo = construir_nodo_supervisor(generador_malo)

try:
    await nodo_malo({"mensajes": [{"role": "user", "content": "hola"}]})
except ErrorSalidaInvalida as exc:
    print("Rechazado como corresponde:", exc)


## 5. Opcional — con un LLM real

El mismo núcleo, sin cambiar nada, corre con el modelo real. Solo se ejecuta si
tienes `LLM_PROVIDER` configurado en el entorno (o en `.env`). El adaptador
`GeneradorLangchain` envuelve cualquier modelo LangChain con
`with_structured_output`, sin nombrar proveedores.


In [ ]:
import os

if os.getenv("LLM_PROVIDER"):
    from src.agentes.generador_langchain import GeneradorLangchain
    from src.llm_provider import get_llm

    generador_real = GeneradorLangchain(get_llm())
    intencion_real = await clasificar_intencion(
        "Quiero un resumen ejecutivo de VCN para un gestor", None, generador_real
    )
    print(intencion_real.model_dump())
else:
    print(
        "Sin LLM_PROVIDER configurado: se omite la prueba real "
        "(no hace falta para el resto del notebook)."
    )


## Qué aprendimos

- El agente **no instancia el LLM**: lo recibe inyectado. Por eso se puede probar
  sin red, sin claves y sin gastar cuota.
- El mismo núcleo corre con un `FakeGenerator` o con el modelo real: cambia la
  inyección, no la lógica.
- El nodo del grafo es una capa fina sobre el núcleo; traduce estado → estado.

**Cómo se extiende:** para sumar otro proveedor, se construye su modelo LangChain
en `app.py` y se envuelve con `GeneradorLangchain`. El Supervisor no se toca.
